# Workshop 3: Neural networks in PyTorch

| Minutes | Activity |
|---|---|
| 0–12 | Part 1: What is a neural network? We talk through it together |
| 12–55 | Part 2: Build one in PyTorch. We write the code together |

## Getting started (2 minutes)

- Save your own copy in Colab (**File → Save a copy in Drive**).
- Click the **play button** beside a code cell to run it. Work from top to bottom.
- Double-click text to edit an answer. Click outside when done.

The first cell loads the libraries:

| Library | What it does |
|---|---|
| `pandas`, `numpy`, `matplotlib.pyplot`, `sklearn` | Tables, maths on many numbers, graphs and ready-made models, as in Workshops 1 and 2 |
| `torch` (PyTorch) | Like NumPy, but it can work out gradients for you and run on a graphics card |
| `torch.nn`, shortened to `nn` | Building blocks for neural networks |

It then loads the California house prices from Workshop 1 into a table called `housing`.

In [ ]:
# Libraries from Workshops 1 and 2.
import pandas as pd  # Stores data in tables.
import numpy as np  # Does maths on many numbers at once.
import matplotlib.pyplot as plt  # Draws graphs.
from sklearn.datasets import fetch_california_housing  # The house prices from Workshop 1.
from sklearn.model_selection import train_test_split  # Splits the data, as in Workshop 1.
from sklearn.preprocessing import StandardScaler  # Rescales inputs, as in Workshop 2.

# New this week: PyTorch.
import torch  # Tensors and automatic gradients.
from torch import nn  # Building blocks for neural networks.

housing = fetch_california_housing(as_frame=True).frame  # Download the data as a table.
housing.head()  # Show the first five rows.

## Part 3: What is a neural network? (15 minutes)

Run each diagram cell and use its controls. The code that draws each diagram is hidden: click **Show code** to see it.

### One neuron

A **neuron** starts like Workshop 1's line. It multiplies its input $x$ by a **weight**, which plays the part of the slope, and adds a **bias**, which plays the part of the intercept.

It then passes the result through an **activation function**. We use **ReLU**, the most common one: it turns negative values into 0 and leaves positive values alone. So the output is flat on one side of the point where the line crosses 0, and a straight line on the other, like a hinge.

$$
\text{output} = \text{ReLU}(\text{weight} \times x + \text{bias}), \qquad \text{ReLU}(z) = \max(0, z)
$$

Drag the sliders. The weight sets how steep the line is and which way it faces, and the bias moves the bend left and right.

In [ ]:
# @title One neuron
from ipywidgets import FloatSlider, interact


@interact(weight=FloatSlider(min=-2, max=2, step=0.1, value=1), bias=FloatSlider(min=-3, max=3, step=0.1, value=0))
def one_neuron(weight, bias):
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))

    left.set(xlim=(0, 10), ylim=(0, 4), title="output = ReLU(weight × x + bias)")
    left.axis("off")
    left.scatter([1.2, 5.2], [2, 2], s=[1500, 4000], color=["white", "tab:blue"], edgecolors="black", zorder=2)
    left.text(1.2, 2, "x", ha="center", va="center", fontsize=16)
    left.text(5.2, 2, f"{'+' if bias >= 0 else '−'} {abs(bias):.1f}\nReLU", ha="center", va="center", fontsize=12, color="white")
    left.annotate("", xy=(4.1, 2), xytext=(1.9, 2), arrowprops={"arrowstyle": "->", "linewidth": 2})
    left.text(3, 2.25, f"× {weight:.1f}".replace("-", "−"), ha="center", fontsize=12)
    left.annotate("", xy=(8.4, 2), xytext=(6.3, 2), arrowprops={"arrowstyle": "->", "linewidth": 2})
    left.text(8.5, 2, "output", va="center", fontsize=12)

    x = np.linspace(-5, 5, 201)
    line = weight * x + bias
    right.plot(x, line, "--", color="grey", label="weight × x + bias")
    right.plot(x, np.maximum(line, 0), color="tab:blue", linewidth=3, label="after ReLU")
    if weight != 0 and abs(bias / weight) <= 5:
        right.scatter([-bias / weight], [0], color="black", zorder=3, label="bend")
    right.axhline(0, color="black", linewidth=0.5)
    right.set(xlabel="x (input)", ylabel="output", ylim=(-5, 5))
    right.legend(loc="upper left")
    plt.show()

PyTorch's ReLU is `torch.relu`. Write a neuron with a weight of 2 and a bias of −4 in one line. Expect `tensor([0., 0., 4., 8., 12., 16.])`.

<details><summary>Hint</summary><code>torch.relu(2 * x - 4)</code></details>

In [ ]:
x = torch.tensor([0.0, 2.0, 4.0, 6.0, 8.0, 10.0])
...  # One neuron: weight 2, bias -4.

Where does this neuron bend?

Your answer: …

### Neurons add up

One neuron on its own cannot fit much. A **neural network** puts several neurons side by side in a **hidden layer**. Each neuron has its own weight and bias, so each one bends in a different place. The output then multiplies each neuron by its own **output weight**, adds them together, and adds a bias of its own.

The three neurons below are already set to fit the dots. Add them one at a time and watch the line change. Each coloured dot on the line marks the bend made by the neuron of the same colour.

In [ ]:
# @title Neurons add up
from IPython.display import Markdown, display
from ipywidgets import Checkbox, ToggleButtons, interact

dots_x = np.arange(0, 10.5, 0.5)
dots_y = 1 + np.maximum(dots_x - 2, 0) - 2 * np.maximum(dots_x - 5, 0) + np.maximum(dots_x - 8, 0)
neurons = [(1, -2, 1), (1, -5, -2), (1, -8, 1)]  # (weight, bias, output weight) for each neuron.
output_bias = 1
colours = ["tab:blue", "tab:orange", "tab:green"]
changes = ["starts to climb", "turns down", "flattens out"]
choices = [("No neurons", 0), ("1 neuron", 1), ("2 neurons", 2), ("3 neurons", 3)]


@interact(added=ToggleButtons(options=choices, description=""), relu=Checkbox(value=True, description="ReLU"))
def neurons_add_up(added, relu):
    def output(x):
        total = np.full(len(x), float(output_bias))
        for weight, bias, output_weight in neurons[:added]:
            line = weight * x + bias
            total = total + output_weight * (np.maximum(line, 0) if relu else line)
        return total

    if added == 0:
        display(Markdown(f"No neurons yet: only the output's bias, a flat line at ${output_bias}$."))
    else:
        weight, bias, output_weight = neurons[added - 1]
        if relu:
            display(Markdown(rf"Neuron {added} adds ${output_weight} \times \text{{ReLU}}({weight} \times x - {-bias})$: the line {changes[added - 1]} at $x = {-bias}$."))
        else:
            display(Markdown(rf"Neuron {added} adds ${output_weight} \times ({weight} \times x - {-bias})$: a straight line, so the output stays straight."))

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))

    left.set(xlim=(0, 10), ylim=(-0.5, 4.5), title="Input → hidden layer → output")
    left.axis("off")
    for n, ((weight, bias, output_weight), colour) in enumerate(zip(neurons, colours)):
        y = 3.5 - 1.5 * n
        alpha = 1 if n < added else 0.15
        label = f"ReLU({weight} × x − {-bias})" if relu else f"{weight} × x − {-bias}"
        left.plot([1, 3.6], [2, y], color="grey", alpha=alpha, zorder=1)
        left.plot([6.4, 9], [y, 2], color=colour, linewidth=1 + abs(output_weight), alpha=alpha, zorder=1)
        left.text(5, y, label, ha="center", va="center", fontsize=10, color="white", alpha=alpha, zorder=2,
                  bbox={"boxstyle": "round,pad=0.5", "facecolor": colour, "edgecolor": "black", "alpha": alpha})
        left.text(7.6, (y + 2) / 2 + 0.3, f"× {output_weight}".replace("-", "−"), ha="center", fontsize=10, alpha=alpha)
    left.scatter([1, 9], [2, 2], s=1600, color="white", edgecolors="black", zorder=2)
    left.text(1, 2, "x", ha="center", va="center", fontsize=14)
    left.text(9, 2, f"+ {output_bias}", ha="center", va="center", fontsize=10)

    x = np.linspace(0, 10, 201)
    right.scatter(dots_x, dots_y, color="grey", s=15, zorder=2)
    right.plot(x, output(x), color="black", linewidth=3, zorder=3)
    if relu:
        for (weight, bias, output_weight), colour in zip(neurons[:added], colours):
            bend = -bias / weight
            right.scatter([bend], output(np.array([bend])), s=120, color=colour, edgecolors="black", zorder=4)
    right.set(xlabel="x (input)", xlim=(0, 10), ylim=(-1, 6), title="Output")
    plt.show()

1. What does each neuron add to the line?
2. Untick **ReLU** and add the neurons again. What happens? Where did you see this in Part 2?

Your answer: …

This is why a network needs an activation function. Without one, a network is a single straight line however many neurons and layers it has, so making it bigger changes nothing. The activation function gives each neuron its bend, and the bends are what let a network follow curves.

The whole network in the diagram fits in one line of code: the output's bias, plus each neuron times its output weight. Write it and check that you get the dots: `tensor([1., 1., 1., 2., 3., 4., 3., 2., 1., 1., 1.])`.

<details><summary>Hint</summary><code>1 + torch.relu(x - 2) - 2 * torch.relu(x - 5) + torch.relu(x - 8)</code></details>

In [ ]:
x = torch.arange(0.0, 11.0)  # 0, 1, 2, ..., 10
...  # The output's bias, plus each neuron times its output weight.

### Layers and parameters

A network can have more than one hidden layer. Each neuron in a layer takes every output of the layer before it. Every line in the diagram below is a weight, and every neuron has a bias. Together, the weights and biases are the network's **parameters**: the numbers that training adjusts.

Drag the sliders and watch the count grow. GPT-3 has 175 billion parameters, and it was trained with the same kind of loop as yours.

In [ ]:
# @title Layers and parameters
from ipywidgets import IntSlider, interact


@interact(inputs=IntSlider(min=1, max=8, value=1), neurons=IntSlider(min=1, max=10, value=3), hidden_layers=IntSlider(min=1, max=3, value=1))
def layers_and_parameters(inputs, neurons, hidden_layers):
    sizes = [inputs] + [neurons] * hidden_layers + [1]
    parameters = sum(before * after + after for before, after in zip(sizes, sizes[1:]))  # Weights, then biases.
    names = ["inputs"] + [f"hidden layer {n + 1}" for n in range(hidden_layers)] + ["output"]
    layers = [[(2 * column, (size - 1) / 2 - row) for row in range(size)] for column, size in enumerate(sizes)]

    fig, ax = plt.subplots(figsize=(11, 4.5))
    for before, after in zip(layers, layers[1:]):
        for x1, y1 in before:
            for x2, y2 in after:
                ax.plot([x1, x2], [y1, y2], color="grey", linewidth=0.7, zorder=1)
    for column, (layer, name) in enumerate(zip(layers, names)):
        colour = "tab:blue" if 0 < column < len(layers) - 1 else "white"
        ax.scatter([x for x, y in layer], [y for x, y in layer], s=250, color=colour, edgecolors="black", zorder=2)
        ax.text(2 * column, -5.6, name, ha="center")
    ax.set(xlim=(-1, 2 * len(sizes) - 1), ylim=(-6, 5), title=f"{parameters:,} parameters")
    ax.axis("off")
    plt.show()

PyTorch can count them for you. `example_network.parameters()` gives every weight and bias, and `p.numel()` counts the numbers in each one. Count the parameters of a network with 8 inputs and two hidden layers of 64 neurons. Expect **4801**.

<details><summary>Hint</summary><code>sum(p.numel() for p in example_network.parameters())</code></details>

In [ ]:
example_network = nn.Sequential(nn.Linear(8, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))
...  # Add up p.numel() for every p in example_network.parameters().

In Workshops 1 and 2 you made lines bend by adding powers such as $x^2$ and $x^3$. You chose the powers, and the powers decided the shape. A network learns its shape instead: training moves every weight and bias until the bends fit the data.

## Part 4: How a network learns (5 minutes)

A network starts with random weights and biases. The loss tells you how wrong they are, but not which way to change each one.

To see how training decides, take a single parameter: the slope of Workshop 1's income line, with the intercept fixed at 0.45. Each slope gives a different loss, and plotted together they make a **bowl**. The best slope, 0.42, sits at the bottom.

Click **Step** to take one step downhill, and keep clicking. Then click **Reset**, set the learning rate to 0.05 and try again: the dot overshoots the bottom and zigzags before it settles. At 0.06 it overshoots further every time.

In [ ]:
# @title Loss bowl
from IPython.display import display
from ipywidgets import Button, FloatSlider, HBox, Layout, Output, VBox

a, b, c = 18.59, -15.54, 3.95  # The loss is a × slope² + b × slope + c, with the intercept fixed at 0.45.
path = [0.0]
rate = FloatSlider(min=0.005, max=0.06, step=0.005, value=0.01, description="learning rate", readout_format=".3f",
                   style={"description_width": "initial"}, layout=Layout(width="350px"))
step_button = Button(description="Step")
reset_button = Button(description="Reset")
picture = Output()


def loss(slope):
    return a * slope ** 2 + b * slope + c


def gradient(slope):
    return 2 * a * slope + b


def draw(*_):
    slope = path[-1]
    following = slope - rate.value * gradient(slope)
    with picture:
        picture.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 3.8))
        slopes = np.linspace(-0.6, 1.4, 201)
        ax.plot(slopes, loss(slopes), color="grey")
        ax.scatter(path[:-1], [loss(s) for s in path[:-1]], color="tab:blue", alpha=0.35, s=20)
        if -0.6 <= slope <= 1.4:
            ax.annotate("", xy=(following, loss(following)), xytext=(slope, loss(slope)),
                        arrowprops={"arrowstyle": "->", "color": "tab:orange", "linewidth": 2})
            ax.scatter([slope], [loss(slope)], color="tab:blue", edgecolors="black", s=80, zorder=3)
            title = f"Step {len(path) - 1}: slope {slope:.2f}, loss {loss(slope):.2f}, gradient {gradient(slope):.1f}".replace("-", "−")
        else:
            title = "The dot has left the bowl: each step overshoots by more than the last."
        ax.set(xlim=(-0.6, 1.4), ylim=(0, 20), xlabel="slope (intercept fixed at 0.45)", ylabel="loss (MSE)", title=title)
        plt.show()


def step(_):
    if -0.6 <= path[-1] <= 1.4:
        path.append(path[-1] - rate.value * gradient(path[-1]))
        draw()


def reset(_):
    path[:] = [0.0]
    draw()


step_button.on_click(step)
reset_button.on_click(reset)
rate.observe(draw, names="value")
display(VBox([HBox([step_button, reset_button, rate]), picture]))
draw()

Each step changes the slope by the learning rate times the gradient:

$$
\text{slope} \leftarrow \text{slope} - \text{learning rate} \times \text{gradient}
$$

- The **gradient** says how steep the bowl is under the dot, and which way is uphill. The step goes the other way, which is why the formula subtracts it. The orange arrow shows the step.
- `loss.backward()` works out the gradient for every parameter at once, however many there are.
- The **optimiser** turns each gradient into a step. `SGD` multiplies it by the learning rate. **Adam** also adapts the size of each parameter's steps as it goes.

Training a network, as `fit` does, is this picture repeated thousands of times, for every weight and bias at once.

Now find a gradient yourself. The cell below makes a line with a random slope and intercept, and measures its loss on the house prices. Add the backward pass. Expect a slope gradient of about **−15.2**.

<details><summary>Hint</summary><code>loss.backward()</code></details>

In [ ]:
incomes = torch.tensor(housing[["MedInc"]].to_numpy(), dtype=torch.float32)
values = torch.tensor(housing[["MedHouseVal"]].to_numpy(), dtype=torch.float32)

torch.manual_seed(0)
line = nn.Linear(1, 1)  # A straight line with a random slope and intercept.
loss = nn.MSELoss()(line(incomes), values)  # How wrong it is.
...  # Find the gradient for every parameter.
print("slope:", line.weight.item(), "| slope gradient:", line.weight.grad.item())

The gradient is negative. Will the next step make the slope bigger or smaller?

Your answer: …

The cell below is the loop inside `fit`, using plain gradient descent (`SGD`) so that the learning rate is the only setting. Run it, then change `learning_rate` to 0.1 and run it again.

In [ ]:
learning_rate = 0.01  # Then try 0.1.

torch.manual_seed(0)
line = nn.Linear(1, 1)
optimizer = torch.optim.SGD(line.parameters(), lr=learning_rate)
for step in range(101):
    optimizer.zero_grad()
    loss = nn.MSELoss()(line(incomes), values)
    loss.backward()
    optimizer.step()
    if step % 20 == 0:
        print("step", step, "| loss", loss.item())

What happens to the loss with a learning rate of 0.1? Why?

Your answer: …

## Part 2: Build a neural network in PyTorch (35 minutes)

### 1. Tensors and gradients (6 minutes)

PyTorch stores numbers in **tensors**. A tensor works like a NumPy array: maths happens on every number at once. The difference is that PyTorch can record every calculation on a tensor, then work backwards through them to find gradients.

We start with [Workshop 1's model](https://warwick.ai/education/courses/ml-ai-flagship/workshop-1#2-fit-a-model-and-predict-5-minutes), with both parameters at 0:

$$
\text{value} = \text{slope} \times \text{income} + \text{intercept}
$$

| Code | What it does |
|---|---|
| `torch.tensor(column.to_numpy(), dtype=torch.float32)` | Turns a column into a tensor of decimal numbers |
| `torch.tensor(0.0, requires_grad=True)` | A parameter: PyTorch records every calculation that uses it |
| `(a ** 2).mean()` | Squares every value, then takes the mean |
| `loss.backward()` | The backward pass: calculates the gradient of the loss for every parameter |
| `slope.grad` | The gradient for `slope` |
| `.item()` | Turns a tensor holding one number into a plain Python number |

Complete the loss, then run the backward pass. Expect a loss of about **5.61**, a slope gradient of about **−19.03** and an intercept gradient of about **−4.14**.

<details><summary>Hint</summary><code>loss = ((predicted - values) ** 2).mean()</code>, then <code>loss.backward()</code>.</details>

In [ ]:
incomes = torch.tensor(housing["MedInc"].to_numpy(), dtype=torch.float32)
values = torch.tensor(housing["MedHouseVal"].to_numpy(), dtype=torch.float32)

slope = torch.tensor(0.0, requires_grad=True)
intercept = torch.tensor(0.0, requires_grad=True)

predicted = slope * incomes + intercept  # Forward pass.
loss = ...  # The MSE: square the errors, then take the mean.
...  # Backward pass.

print("loss:", loss.item())
print("slope gradient:", slope.grad.item(), "| intercept gradient:", intercept.grad.item())

Both gradients are negative. Will gradient descent make `slope` and `intercept` bigger or smaller? Why?

Your answer: …

### 2. The training loop (8 minutes)

PyTorch has ready-made pieces for the rest of the loop. `nn.Linear(1, 1)` is Workshop 1's straight line: one input and one output. PyTorch calls its slope the **weight** (`model.weight`) and its intercept the **bias** (`model.bias`), and starts them at random values. An **optimiser** does the gradient descent step for every parameter. `SGD` is plain gradient descent.

| Code | What it does |
|---|---|
| `housing[["MedInc"]]` | A one-column table. Models take their inputs as a table, as in Workshop 1 |
| `nn.Linear(1, 1)` | A straight line with one input and one output |
| `nn.MSELoss()` | The MSE loss function |
| `torch.optim.SGD(model.parameters(), lr=0.01)` | Gradient descent on every parameter of `model`, with a learning rate of 0.01 |
| `optimizer.zero_grad()` | Clears the gradients from the last step |
| `optimizer.step()` | Moves every parameter one step downhill |

Fill in the four steps from the recap table. Expect a slope of about **0.42**, an intercept of about **0.45** and a loss of about **0.70**: the same line as `LinearRegression` found in Workshop 1.

<details><summary>Hint</summary><code>predicted = model(inputs)</code>, <code>loss = loss_fn(predicted, targets)</code>, <code>loss.backward()</code>, <code>optimizer.step()</code>.</details>

In [ ]:
inputs = torch.tensor(housing[["MedInc"]].to_numpy(), dtype=torch.float32)
targets = torch.tensor(housing[["MedHouseVal"]].to_numpy(), dtype=torch.float32)

torch.manual_seed(0)  # Start from the same random line every time.
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for step in range(1000):
    optimizer.zero_grad()  # Clear the old gradients.
    predicted = ...  # Forward pass.
    loss = ...  # Loss.
    ...  # Backward pass.
    ...  # Step downhill.

print("slope:", model.weight.item(), "| intercept:", model.bias.item(), "| loss:", loss.item())

These five lines train every neural network. For a bigger model, only the `model = ...` line changes.

Delete the `optimizer.zero_grad()` line and run the cell again. What happens to the slope and the loss? Why? Put the line back afterwards.

Your answer: …

### 3. From a line to a network (10 minutes)

House values go up and down with latitude, from San Diego in the south (33°) to the Oregon border (42°). A straight line cannot follow that. A network with a hidden layer of ReLU neurons can, because each neuron adds a bend, as in Part 1.

The cell below rescales latitude with `StandardScaler`, as in [Workshop 2](https://warwick.ai/education/courses/ml-ai-flagship/workshop-2#2-see-overfitting), which helps training. It also defines `draw`, which plots a network's predictions over the houses.

In [ ]:
latitude_scaler = StandardScaler()
latitudes = torch.tensor(latitude_scaler.fit_transform(housing[["Latitude"]].to_numpy()), dtype=torch.float32)
sample = housing.sample(2000, random_state=0)  # Plot 2,000 houses so the graph draws quickly.


def draw(network):
    grid = np.linspace(32.5, 42, 400).reshape(-1, 1)  # 400 latitudes from south to north.
    with torch.no_grad():  # Only predicting, so PyTorch does not need to record anything.
        curve = network(torch.tensor(latitude_scaler.transform(grid), dtype=torch.float32))
    plt.scatter(sample["Latitude"], sample["MedHouseVal"], s=3, alpha=0.3)
    plt.plot(grid, curve, color="black")
    plt.xlabel("Latitude (degrees north)")
    plt.ylabel("Median house value ($100,000s)")
    plt.show()

Write a `fit` function that trains any network, like scikit-learn's `fit`. Its loop is the five lines from section 2.

It uses `torch.optim.Adam` in place of `SGD`. **Adam** is gradient descent that adapts the step size for each parameter. It trains most neural networks.

<details><summary>Hint</summary>Copy the five lines inside the loop from section 2, and change <code>model</code> to <code>network</code>.</details>

In [ ]:
def fit(network, inputs, targets, steps=2000):
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(network.parameters(), lr=0.01)
    for step in range(steps):
        ...  # The five lines from section 2.
    return loss.item()  # The final training loss.

`nn.Sequential` runs layers in order. Complete the network: 50 neurons, ReLU, then add them up into one prediction. Expect a loss of about **1.10**. A straight line gets about **1.30**.

<details><summary>Hint</summary><code>nn.ReLU()</code></details>

In [ ]:
neurons = 50

torch.manual_seed(0)
network = nn.Sequential(
    nn.Linear(1, neurons),  # Each neuron: weight × latitude + bias.
    ...,  # Bend every neuron.
    nn.Linear(neurons, 1),  # Add up the neurons into one prediction.
)
print("loss:", fit(network, latitudes, targets))
draw(network)

Los Angeles is at about 34° and San Francisco at about 37.8°. What has the network found?

Your answer: …

1. Change `neurons` to 5 and run the cell again. Why is the fit worse?
2. Set `neurons` back to 50, then delete the `nn.ReLU()` line. What shape do you get? Why?

Your answer: …

### 4. All 8 inputs (10 minutes)

Now the task from Workshop 1: predict house values from all 8 inputs. We use the same training and validation sets as [Workshop 1's take-home](https://warwick.ai/education/courses/ml-ai-flagship/workshop-1#take-home-find-the-sweet-spot-about-15-minutes), so we can compare scores. The test set stays aside for one final check.

In [ ]:
train, rest = train_test_split(housing, test_size=0.3, random_state=0)
val, test = train_test_split(rest, test_size=0.5, random_state=0)
input_columns = housing.columns.drop("MedHouseVal")  # The names of the 8 input columns.

scaler = StandardScaler()  # Learns each column's average and spread from the training set only.
X_train = torch.tensor(scaler.fit_transform(train[input_columns]), dtype=torch.float32)
X_val = torch.tensor(scaler.transform(val[input_columns]), dtype=torch.float32)
y_train = torch.tensor(train[["MedHouseVal"]].to_numpy(), dtype=torch.float32)
y_val = torch.tensor(val[["MedHouseVal"]].to_numpy(), dtype=torch.float32)
print(X_train.shape, X_val.shape)  # (rows, inputs)

`fit_and_validate` is `fit` with two additions: after every step it also records the loss on the validation set, and at the end it plots both losses. Complete the line that records the validation loss, then run the cell.

<details><summary>Hint</summary><code>loss_fn(network(X_val), y_val).item()</code></details>

In [ ]:
def fit_and_validate(network, steps=1000, weight_decay=0):  # Extension B uses weight_decay.
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(network.parameters(), lr=0.003, weight_decay=weight_decay)  # A smaller learning rate gives smoother curves.
    train_losses = []
    val_losses = []
    for step in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(network(X_train), y_train)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        with torch.no_grad():  # Only measuring, not training.
            val_losses.append(...)  # The loss on the validation set.

    best_step = np.argmin(val_losses)
    print("final training MSE:", round(train_losses[-1], 3), "| final validation MSE:", round(val_losses[-1], 3))
    print("lowest validation MSE:", round(val_losses[best_step], 3), "at step", best_step)
    plt.plot(train_losses, label="Training")
    plt.plot(val_losses, label="Validation")
    plt.ylim(0, 1.5)
    plt.xlabel("Step")
    plt.ylabel("MSE")
    plt.legend()
    plt.show()

This network has two hidden layers of 64 neurons. How many inputs does its first layer take? Expect a training MSE of about **0.23** and a validation MSE of about **0.29**.

<details><summary>Hint</summary>Each house has 8 inputs: <code>nn.Linear(8, 64)</code>.</details>

In [ ]:
torch.manual_seed(0)
network = nn.Sequential(
    nn.Linear(..., 64),  # How many inputs does each house have?
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
)
fit_and_validate(network)

Compare your network with the models from Workshop 1, all scored on the same validation set. (Workshop 1's 0.52 for linear regression was scored on the data it trained on.)

| Model | Validation MSE |
|---|---|
| Always predict the average | 1.33 |
| Linear regression, 8 inputs | 0.54 |
| Decision tree, depth 9 | 0.43 |
| Random forest | 0.28 |
| Your network | |

How does it do? Is it overfitting yet?

Your answer: …

### Lab check (1 minute)

Explain in one sentence each:

- What `loss.backward()` does.
- Why a neural network needs ReLU between its layers.

Your answer: …

### Extension A: Make it bigger

Give the network three hidden layers of 256 neurons each: 134,145 parameters. It takes about half a minute to train.

<details><summary>Hint</summary>Each extra hidden layer is <code>nn.Linear(256, 256)</code>, then <code>nn.ReLU()</code>.</details>

In [ ]:
torch.manual_seed(0)
big_network = nn.Sequential(
    nn.Linear(8, 256),
    nn.ReLU(),
    ...,  # Two more hidden layers of 256 neurons, each followed by ReLU.
    nn.Linear(256, 1),
)
fit_and_validate(big_network)

What happens to the two losses? When would you stop training?

Your answer: …

### Extension B: Weight decay is Ridge

[Workshop 2's Ridge](https://warwick.ai/education/courses/ml-ai-flagship/workshop-2#3-ridge-shrink-all-the-weights) added a penalty for big weights to the loss. In PyTorch, `weight_decay` on the optimiser adds the same penalty. Run the cell to train the big network again with `weight_decay=1e-3`.

In [ ]:
torch.manual_seed(0)
big_network = nn.Sequential(
    nn.Linear(8, 256),
    nn.ReLU(),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Linear(256, 1),
)
fit_and_validate(big_network, weight_decay=1e-3)

What changes compared with Extension A?

Your answer: …